# 50 — Materialize Fabric gold → Supabase Postgres

**Why:** the web backend (Azure Container Apps) cannot reach the Fabric SQL endpoint,
so custom React pages (compliance, portfolio KPIs) show *"Live data unavailable"*.
This notebook copies the gold columns the backend needs into Supabase `mv_` tables,
which the backend CAN reach. Run it after the Fabric pipeline (manually now; schedule later).

**Before running:** create the `mv_` tables once (the backend's alembic migration
`fab1c0d3e5a7` does this automatically on its next deploy; or apply it manually).

**Fill the password** in the Parameters cell from Supabase → Project Settings → Database →
Connection string. Do NOT commit the password.


## Parameters — fill PG_PASSWORD, then Run All


In [ ]:
# Supabase Postgres connection (Project Settings -> Database -> Connection)
PG_HOST = "db.vuiqfaklvlbushkiapnp.supabase.co"  # or session-pooler host if direct is blocked
PG_PORT = 5432
PG_DB   = "postgres"
PG_USER = "postgres"                 # pooler user is 'postgres.vuiqfaklvlbushkiapnp'
PG_PASSWORD = ""                      # <-- PASTE your Supabase DB password (do not commit)

# Limit to showcase buildings, or None for ALL buildings in the model.
BUILDING_IDS = None                    # e.g. ["B001","B002",...,"B011"]

assert PG_PASSWORD, "Set PG_PASSWORD from Supabase before running."


In [ ]:
%pip install psycopg2-binary --quiet


## Helper — truncate + bulk-insert a Spark DataFrame into a Postgres table


In [ ]:
import math
from datetime import datetime, timezone
import psycopg2
from psycopg2.extras import execute_values

def _conn():
    return psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB,
                            user=PG_USER, password=PG_PASSWORD, sslmode="require")

def _clean(v):
    # NaN/NaT -> None so Postgres stores NULL
    if v is None:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    return v

def materialize(df, table, columns):
    pdf = df.select(*columns).toPandas()
    rows = [tuple(_clean(v) for v in rec)
            for rec in pdf.itertuples(index=False, name=None)]
    conn = _conn()
    try:
        cur = conn.cursor()
        cur.execute(f"TRUNCATE TABLE {table}")
        if rows:
            collist = ", ".join(columns)
            execute_values(cur,
                f"INSERT INTO {table} ({collist}) VALUES %s", rows, page_size=1000)
        cur.execute(
            "INSERT INTO mv_sync_log (table_name, last_synced_at, row_count) "
            "VALUES (%s, %s, %s) ON CONFLICT (table_name) DO UPDATE SET "
            "last_synced_at = EXCLUDED.last_synced_at, row_count = EXCLUDED.row_count",
            (table, datetime.now(timezone.utc), len(rows)))
        conn.commit()
        print(f"{table}: {len(rows)} rows materialized")
    except Exception:
        conn.rollback(); raise
    finally:
        conn.close()

def _where():
    if not BUILDING_IDS:
        return ""
    ids = ", ".join("'" + b.replace("'", "''") + "'" for b in BUILDING_IDS)
    return f"WHERE building_id IN ({ids})"


## 1) silver_building_master → mv_building_master


In [ ]:
df = spark.sql(f"""
    SELECT building_id, building_name, building_type, gross_floor_area_m2,
           city, country_code, energy_certificate
    FROM silver_building_master {_where()}
""")
materialize(df, "mv_building_master",
    ["building_id","building_name","building_type","gross_floor_area_m2",
     "city","country_code","energy_certificate"])


## 2) gold_ghg_scope → mv_ghg_scope (monthly rows; backend sums per year)


In [ ]:
df = spark.sql(f"""
    SELECT building_id, reporting_year,
           scope1_total_tco2, scope2_location_tco2, scope2_market_tco2,
           scope3_estimated_tco2, total_ghg_location_tco2, total_ghg_market_tco2,
           data_quality_flag
    FROM gold_ghg_scope {_where()}
""")
materialize(df, "mv_ghg_scope",
    ["building_id","reporting_year","scope1_total_tco2","scope2_location_tco2",
     "scope2_market_tco2","scope3_estimated_tco2","total_ghg_location_tco2",
     "total_ghg_market_tco2","data_quality_flag"])


## 3) gold_kpi_daily → mv_kpi_daily (energy + solar self-consumption)


In [ ]:
df = spark.sql(f"""
    SELECT building_id, date, total_consumption_kwh, solar_self_consumed_kwh
    FROM gold_kpi_daily {_where()}
""")
materialize(df, "mv_kpi_daily",
    ["building_id","date","total_consumption_kwh","solar_self_consumed_kwh"])


## Done — verify


In [ ]:
conn = _conn(); cur = conn.cursor()
cur.execute("SELECT table_name, row_count, last_synced_at FROM mv_sync_log ORDER BY table_name")
for r in cur.fetchall(): print(r)
conn.close()
